# temperature regimes

Sample a thermal-regime catalog from an explicitly synthetic Calor dataset.

In [ ]:
from tdpy.verbosity import print

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys

from IPython.display import FileLink, Image, SVG, display

relative_script = Path('examples/temperature_regimes.py')
repository_directory = Path('astromusers/calor')
working_directory = Path.cwd().resolve()
repository_path = next(
    (root for candidate in (working_directory, *working_directory.parents)
     for root in (candidate, candidate / repository_directory)
     if (root / '.git').exists() and (root / relative_script).is_file()),
    None,
)
if repository_path is None:
    raise RuntimeError('Open this notebook from its checkout or the parent workspace')
script_path = repository_path / relative_script
arguments = []


In [ ]:
environment = os.environ.copy()
environment.setdefault('CALOR_PATH', str(repository_path))
python_paths = [str(repository_path), str(repository_path / 'src')]
if environment.get('PYTHONPATH'):
    python_paths.append(environment['PYTHONPATH'])
environment['PYTHONPATH'] = os.pathsep.join(python_paths)
print(f'Reading from {script_path}...')
completed = subprocess.run(
    [sys.executable, str(script_path), *arguments],
    cwd=repository_path, env=environment, capture_output=True, text=True,
)
print(completed.stdout, end='')
print(completed.stderr, end='', file=sys.stderr)
completed.check_returncode()


In [ ]:
import argparse

output_parser = argparse.ArgumentParser(add_help=False)
output_parser.add_argument('--figure-path', type=Path)
output_parser.add_argument('--typefileplot', choices=('png', 'pdf'), default='png')
output_options, _ = output_parser.parse_known_args(arguments)
calor_path = Path(environment.get('CALOR_PATH') or repository_path).expanduser().resolve()
expected_figure_path = output_options.figure_path or calor_path / 'visuals' / f'temperature_regimes.{output_options.typefileplot}'
if not expected_figure_path.is_absolute():
    expected_figure_path = repository_path / expected_figure_path

figure_paths = []
for filename in re.findall(r'^Writing to (.+)\.\.\.$', completed.stdout, flags=re.MULTILINE):
    figure_path = Path(filename)
    if not figure_path.is_absolute():
        figure_path = repository_path / figure_path
    if figure_path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.gif', '.svg', '.pdf'}:
        if figure_path.is_file() and figure_path not in figure_paths:
            figure_paths.append(figure_path)
if expected_figure_path.is_file() and expected_figure_path not in figure_paths:
    figure_paths.append(expected_figure_path)
for figure_path in figure_paths:
    print(f'Reading from {figure_path}...')
    if figure_path.suffix.lower() == '.svg':
        display(SVG(filename=str(figure_path)))
    elif figure_path.suffix.lower() == '.pdf':
        display(FileLink(os.path.relpath(figure_path, Path.cwd())))
    else:
        display(Image(filename=str(figure_path)))